In [50]:
from dotenv import load_dotenv
load_dotenv()

True

In [51]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import  GoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [52]:
loader=PyPDFLoader("../data/medical_report.pdf")
docs=loader.load()


In [53]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_docs=splitter.split_documents(docs)


In [54]:
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embeddings
)

In [55]:
same_record=vector_store.similarity_search("Pateint name")

### Agent=Tools llm Prompt

In [56]:
@tool
def retriever_tool(query:str):
    """
        ALWAYS use this tool to answer any question related to the medical report.
    It retrieves relevant content from the medical report PDF.
    """
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context+=doc.page_content + "\n\n"
    return context

    

In [57]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [58]:
system_prompt = """
You are a medical assistant.

You MUST use the retriever_tool to answer any question about the medical report.
Do NOT answer from your own knowledge.

Only return answers based on the retrieved context.
If the answer is not found, say "Not found in report".
"""

In [59]:
agent=create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [60]:
query="What is the name of patient? and what is the name of the doctor?"
res=agent.invoke({"messages":[{"role":"user","content":query}]})

In [61]:
result=res["messages"][-1].content

In [62]:
print(result)

The patient's name is Ms. NIKITA CHUDHARY. The doctor's name is DR NITIN NAHAR.
